# Diffusion-Denoised Spatiotemporal GNN (DS-TGNN) Walkthrough

This notebook demonstrates the end-to-end flow of our DS-TGNN pipeline for commodity excess return prediction. We'll trace a single batch of data as it physically moves through each step of the architecture: from noisy weather inputs through the GNN, ending at the predicted scalars.


In [ ]:
import torch
import sys
sys.path.append('models')

from dataset import get_dataloaders
from ds_tgnn import DiffusionReturnPrediction
from diffusion.diffusion_architecture import Diffusion
from diffusion.loss_func import ScoreDiffusionLoss
from ds_tgnn_train import build_supply_chain_graph


## 1. Input Data & Graph Construction
First, we define our environment. We track 8 commodities across 180 days with 5 weather features each.
We also build our static, directed Supply Chain Graph which guides the spatial message passing.

In [ ]:
# Config
N = 8          # Commodities
T_days = 180   # Time window
F = 5          # Weather features
batch_size = 4

# Get dummy data batch
train_loader, _ = get_dataloaders(batch_size=batch_size, N=N, T=T_days, F=F)
batch_x, batch_targets = next(iter(train_loader))

print(f"Raw Weather Input Shape: {batch_x.shape} -> [Batch, Nodes, Time, Features]")
print(f"Target Returns Shape:    {batch_targets.shape} -> [Batch, Nodes]\n")

# Construct PyG Supply Chain Graph
edge_index = build_supply_chain_graph()
print("Supply Chain Edges [Source -> Target]:")
print(edge_index)
print(f"Number of directed edges: {edge_index.shape[1]}")


## 2. Instantiating the Models
We instantiate the **Denoiser** (`Diffusion`), the **Predictor** (`DiffusionReturnPrediction`), and the **Loss function** (`ScoreDiffusionLoss`).

In [ ]:
# The underlying denoiser expects flattened shapes given our MLP architecture choice.
input_dim_flat = N * T_days * F  

denoiser = Diffusion(
    input_dim=input_dim_flat, 
    mlp_hidden=128, 
    conv_hidden=32, 
    t_hidden_dim=16, 
    output_dim=input_dim_flat, 
    use_conv=False
)

score_loss_fn = ScoreDiffusionLoss(sigma_min=1e-4, sigma_max=1.0)

model = DiffusionReturnPrediction(
    denoiser=denoiser,
    input_dim=F,
    lstm_hidden=64,
    gnn_hidden=32
)

print("Models instantiated gracefully.")


## 3. Step 1: Objective 1 - Score Matching (Denoiser Training)
The diffusion denoiser learns the fundamental manifold of regional weather interactions by attempting to reconstruct original weather sequences from Gaussian noise.
This does not backpropagate through the downstream predictor or GNN!

In [ ]:
# Flatten the spatial/temporal dims for the underlying MLP Denoiser
x_clean_flat = batch_x.view(batch_size, -1)
print(f"Flattened Input for Denoiser: {x_clean_flat.shape}")

# Calculate internal Noising and Score-Loss (MSE)
loss_diff = score_loss_fn(model.denoiser, x_clean_flat)
print(f"\nObjective 1 - Diffusion Score Loss: {loss_diff.item():.4f}")


## 4. Step 2: Extracting Denoised Features
For the supervised return-prediction pipeline, we do *not* run the massive iterative ODE reversal solver during training. Instead, we generate a highly-conditional "1-step denoised expectation" from slightly-noised data to feed into the LSTM.

In [ ]:
import copy

with torch.no_grad():
    # Generate fractional continuous timestep (e.g. 10% into the forward schedule)
    t_low = torch.full((batch_size,), 0.1)
    sigma_low = score_loss_fn.sigma_min * (score_loss_fn.sigma_max / score_loss_fn.sigma_min) ** t_low
    sigma_low_view = sigma_low.view(batch_size, 1)

    # Corrupt clean sequence
    x_noisy_flat = x_clean_flat + torch.randn_like(x_clean_flat) * sigma_low_view

# Use the Denoiser to estimate the applied noise Z
z_pred_flat = model.denoiser(x_noisy_flat, sigma_low.view(-1, 1))

# Subtract the predicted noise (scaled by sigma) to arrive at the 1-Step estimated clean representation
denoised_repr_flat = x_noisy_flat - z_pred_flat * sigma_low_view 

# Finally, restructure it back to independent sequence nodes for LSTM processing
denoised_repr = denoised_repr_flat.view(batch_size, N, T_days, F)

print(f"Denoised Spatial-Temporal Features: {denoised_repr.shape}")


## 5. Step 3 & 4: Objective 2 - Spatiotemporal Supervised Return Prediction
Now we pass the $T=180$ weather steps into the exact pipeline request:
1. **LSTM:** Compresses each commodity's 180-day series `[T=180, F=5]` into a single context node `[H=64]`.
2. **GNN:** Allows message passing along the supply chain. Corn sends state updates to Cattle/Hogs.
3. **Shared MLP:** A single Dense layer evaluates the enriched sequence embeddings and assigns an expected PnL percentage.

In [ ]:
predicted_returns = model(denoised_repr, edge_index)

print(f"Final Predicted Scalar Returns Shape: {predicted_returns.shape} -> [Batch, Nodes]")
print("Sample Predictions:")
print(predicted_returns.detach())

loss_pred = torch.nn.functional.mse_loss(predicted_returns, batch_targets)
print(f"\nObjective 2 - Predictive MSE Loss: {loss_pred.item():.4f}")


## Next steps
In full operation, `opt_diff.step()` and `opt_pred.step()` happen synchronously with these outputs exactly as demonstrated here!